# BERT WSI

Linear, function-free Word Sense Induction experiment using RuBERT contextual embeddings for the target word.

In [1]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

c:\Users\stla7\OneDrive\Рабочий стол\ML\NLP\pet_first\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
df = pd.read_csv("data/train/train.csv", sep="\t")
df = df.drop(columns=["predict_sense_id"], errors="ignore")

print(f"Rows: {len(df)}")
print(f"Words: {df['word'].nunique()}")
print(f"Missing positions: {df['positions'].isna().sum()}")
df.head()

Rows: 2073
Words: 85
Missing positions: 33


,context_id,word,gold_sense_id,positions,context
0,1,дар,1,18-22,Отвергнуть щедрый дар
1,2,дар,1,21-28,покупать преданность дарами и наградами
2,3,дар,1,19-23,Вот яд – последний дар моей Изоры
3,4,дар,1,81-87,Основная функция корильных песен – повеселить ...
4,5,дар,1,151-157,Но недели две спустя (Алевтина его когда-то об...


## Validation Helpers

In [3]:
random_states = [42, 7, 13, 21, 100]
valid_frac = 0.2

def make_validation_split(df, random_state, valid_frac=0.2):
    rng = np.random.RandomState(random_state)
    train_idx = []
    valid_idx = []

    for _, word_df in df.groupby("word"):
        for _, sense_df in word_df.groupby("gold_sense_id"):
            indices = sense_df.index.to_numpy().copy()
            rng.shuffle(indices)

            if len(indices) < 2:
                train_idx.extend(indices.tolist())
                continue

            n_valid = max(1, int(round(len(indices) * valid_frac)))
            n_valid = min(n_valid, len(indices) - 1)

            valid_idx.extend(indices[:n_valid].tolist())
            train_idx.extend(indices[n_valid:].tolist())

    train_df = df.loc[sorted(train_idx)].copy()
    valid_df = df.loc[sorted(valid_idx)].copy()
    return train_df, valid_df

random_state = random_states[0]
train_df, valid_df = make_validation_split(df, random_state, valid_frac)

print(f"Seeds: {random_states}")
print(f"Preview seed: {random_state}")
print(f"Train rows: {len(train_df)}")
print(f"Valid rows: {len(valid_df)}")

Seeds: [42, 7, 13, 21, 100]
Preview seed: 42
Train rows: 1662
Valid rows: 411


## Resolve Target Positions

In [4]:
target_token_pattern = re.compile(r"[A-Za-z\u0410-\u042f\u0430-\u044f\u0401\u0451-]+")

def resolve_target_positions(train_df, valid_df):
    resolved = {}

    for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
        split_df = split_df.copy()
        resolved_positions = []

        for row in split_df.itertuples():
            position_value = np.nan
            positions_str = "" if pd.isna(row.positions) else str(row.positions)

            for chunk in positions_str.split(","):
                chunk = chunk.strip()
                if "-" not in chunk:
                    continue
                start_text, end_text = chunk.split("-", 1)
                try:
                    start_char = int(start_text)
                    end_char = int(end_text)
                    position_value = f"{start_char}-{end_char}"
                    break
                except ValueError:
                    pass

            if pd.isna(position_value):
                text_lower = str(row.context).lower()
                word_lower = str(row.word).lower()
                exact_matches = [match.span() for match in re.finditer(re.escape(word_lower), text_lower)]
                if len(exact_matches) == 1:
                    start_char, end_char = exact_matches[0]
                    position_value = f"{start_char}-{end_char}"

            if pd.isna(position_value):
                word_lower = str(row.word).lower()
                token_matches = []
                for match in target_token_pattern.finditer(str(row.context)):
                    token = match.group(0)
                    prefix_len = 0
                    for token_char, word_char in zip(token.lower(), word_lower):
                        if token_char != word_char:
                            break
                        prefix_len += 1
                    min_prefix = max(3, min(len(token), len(word_lower)) - 2)
                    if prefix_len >= min_prefix:
                        token_matches.append(match.span())

                if len(token_matches) == 1:
                    start_char, end_char = token_matches[0]
                    position_value = f"{start_char}-{end_char}"

            resolved_positions.append(position_value)

        split_df["resolved_positions"] = resolved_positions
        failed_rows = split_df[split_df["resolved_positions"].isna()][["context_id", "word", "context"]].copy()
        bert_df = split_df[split_df["resolved_positions"].notna()].copy()
        bert_df["positions"] = bert_df["resolved_positions"]
        bert_df = bert_df.drop(columns=["resolved_positions"])
        resolved[split_name] = (bert_df, failed_rows)

    train_bert_df, train_failed = resolved["train"]
    valid_bert_df, valid_failed = resolved["valid"]
    return train_bert_df, valid_bert_df, train_failed, valid_failed

train_bert_df, valid_bert_df, train_failed, valid_failed = resolve_target_positions(train_df, valid_df)

print(f"Train unresolved rows: {len(train_failed)}")
print(f"Valid unresolved rows: {len(valid_failed)}")
print(f"Train BERT rows: {len(train_bert_df)}")
print(f"Valid BERT rows: {len(valid_bert_df)}")

Train unresolved rows: 16
Valid unresolved rows: 4
Train BERT rows: 1646
Valid BERT rows: 407


## Load BERT

In [5]:
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(random_states[0])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()

DEVICE

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 8306.75it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'cpu'

## Target Embedding Helper

In [6]:
def build_target_embeddings(bert_df, tokenizer, bert_model, device):
    vectors = []

    for row in tqdm(bert_df.itertuples(index=False), total=len(bert_df)):
        start_char = None
        end_char = None
        for chunk in str(row.positions).split(","):
            chunk = chunk.strip()
            if "-" not in chunk:
                continue
            start_text, end_text = chunk.split("-", 1)
            start_char = int(start_text)
            end_char = int(end_text)
            break

        encoded = tokenizer(
            str(row.context),
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True,
            max_length=512,
        )
        offset_mapping = encoded.pop("offset_mapping")[0].tolist()
        model_inputs = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = bert_model(**model_inputs)

        hidden = outputs.last_hidden_state[0]
        token_indices = []

        for token_idx, (token_start, token_end) in enumerate(offset_mapping):
            if token_end <= token_start:
                continue
            if token_end <= start_char:
                continue
            if token_start >= end_char:
                continue
            token_indices.append(token_idx)

        if len(token_indices) == 0:
            token_indices = [1]

        vectors.append(hidden[token_indices].mean(dim=0).detach().cpu().numpy())

    if len(vectors) == 0:
        return np.empty((0, bert_model.config.hidden_size), dtype=np.float32)
    return np.vstack(vectors)

print(f"Embedding size: {bert_model.config.hidden_size}")

Embedding size: 312


## Split Scoring Helper

In [7]:
def score_bert_split(random_state):
    train_df, valid_df = make_validation_split(df, random_state, valid_frac)
    train_bert_df, valid_bert_df, train_failed, valid_failed = resolve_target_positions(train_df, valid_df)

    x_train = build_target_embeddings(train_bert_df, tokenizer, bert_model, DEVICE)
    x_valid = build_target_embeddings(valid_bert_df, tokenizer, bert_model, DEVICE)

    predictions = pd.Series(index=valid_bert_df.index, dtype="object")

    for word in sorted(valid_bert_df["word"].unique()):
        train_mask = (train_bert_df["word"] == word).to_numpy()
        valid_mask = (valid_bert_df["word"] == word).to_numpy()
        y_train = train_bert_df.loc[train_mask, "gold_sense_id"].astype(str)

        if y_train.empty:
            pred = np.repeat("__missing_train__", valid_mask.sum())
        elif y_train.nunique() == 1:
            pred = np.repeat(y_train.iloc[0], valid_mask.sum())
        else:
            clf = LogisticRegression(max_iter=1000, random_state=random_state)
            clf.fit(x_train[train_mask], y_train)
            pred = clf.predict(x_valid[valid_mask])

        predictions.loc[valid_bert_df.index[valid_mask]] = pred

    scored_df = valid_bert_df[["word", "gold_sense_id"]].copy()
    scored_df["prediction"] = predictions.astype(str)
    scored_df["gold_sense_id"] = scored_df["gold_sense_id"].astype(str)

    run_word_scores = []
    for word, word_df in scored_df.groupby("word"):
        ari = adjusted_rand_score(word_df["gold_sense_id"], word_df["prediction"])
        run_word_scores.append((word, ari, len(word_df)))

    word_scores_df = pd.DataFrame(run_word_scores, columns=["word", "ari", "n_valid"])
    ari = word_scores_df["ari"].mean()
    word_scores_df["model"] = "bert_target_embedding"
    word_scores_df["random_state"] = random_state

    run = {
        "model": "bert_target_embedding",
        "random_state": random_state,
        "ari": ari,
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "train_bert_rows": len(train_bert_df),
        "valid_bert_rows": len(valid_bert_df),
        "train_unresolved": len(train_failed),
        "valid_unresolved": len(valid_failed),
    }
    return run, word_scores_df

score_bert_split

<function __main__.score_bert_split(random_state)>

## Run Repeated Validation

In [8]:
bert_runs = []
bert_word_scores = []

for random_state in random_states:
    print(f"=== random_state={random_state} ===")
    run, scores_df = score_bert_split(random_state)
    bert_runs.append(run)
    bert_word_scores.append(scores_df)
    print(f"ARI: {run['ari']:.4f}")

bert_runs_df = pd.DataFrame(bert_runs)
bert_word_scores_df = pd.concat(bert_word_scores, ignore_index=True)

bert_summary_df = (
    bert_runs_df.groupby("model", as_index=False)
    .agg(
        ari=("ari", "mean"),
        ari_std=("ari", "std"),
        runs=("ari", "count"),
        train_unresolved=("train_unresolved", "mean"),
        valid_unresolved=("valid_unresolved", "mean"),
    )
    .sort_values("ari", ascending=False)
)
bert_summary_df

=== random_state=42 ===


100%|██████████| 407/407 [00:01<00:00, 280.07it/s]


ARI: 0.3188
=== random_state=7 ===


100%|██████████| 410/410 [00:01<00:00, 276.26it/s]


ARI: 0.2778
=== random_state=13 ===


...

100%|██████████| 404/404 [00:03<00:00, 117.70it/s]


ARI: 0.2660
=== random_state=100 ===


100%|██████████| 409/409 [00:03<00:00, 111.74it/s]


ARI: 0.2934


,model,ari,ari_std,runs,train_unresolved,valid_unresolved
0,bert_target_embedding,0.302869,0.036779,5,16.0,4.0
